# Telephone Survey Problem with AMPL in Python
## AMPL Modeling for Problem 5

## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

```python
%pip install amplpy
```

In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

In [2]:
%pip install amplpy
from amplpy import AMPL, modules

modules.install('highs')


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
def create_ampl_instance(solver: str = "highs"):
    ampl = AMPL()
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl

## Problem: Telephone Survey Cost Minimization

A market research group must reach at least:
- 150 wives, 120 husbands, 100 single males, 110 single females

**Call costs:** $2 (day), $5 (night)

**Contact probabilities:**

| Person       | Day  | Night |
|--------------|------|-------|
| Wife         | 30%  | 30%   |
| Husband      | 10%  | 30%   |
| Single Male  | 10%  | 15%   |
| Single Female| 10%  | 20%   |
| Nobody       | 40%  | 5%    |

**Constraint:** At most half of all calls can be at night.

Minimize total survey cost.

In [4]:
@timed
def solve_survey(solver: str = "highs"):
    ampl = create_ampl_instance(solver)

    ampl.eval(
        r'''
        var day >= 0;
        var night >= 0;

        minimize TotalCost:
            2 * day + 5 * night;

        subject to Wives:
            0.30 * day + 0.30 * night >= 150;

        subject to Husbands:
            0.10 * day + 0.30 * night >= 120;

        subject to SingleMales:
            0.10 * day + 0.15 * night >= 100;

        subject to SingleFemales:
            0.10 * day + 0.20 * night >= 110;

        subject to NightLimit:
            night <= day;
        '''
    )
    ampl.solve(solver=solver)

    solution = {
        "day": round(ampl.get_value("day"), 4),
        "night": round(ampl.get_value("night"), 4),
        "cost": round(ampl.get_value("TotalCost"), 4),
    }
    return solution

In [5]:
result, elapsed = solve_survey()

print("=== TELEPHONE SURVEY RESULTS ===")
print(f"Daytime calls   -> {result['day']:.2f}")
print(f"Nighttime calls -> {result['night']:.2f}")
print(f"Total calls     -> {result['day'] + result['night']:.2f}")
print(f"Minimum cost    -> ${result['cost']:.2f}")
print(f"Time            -> {elapsed:.8f} seconds")
print()
print("Expected contacts:")
d, n = result["day"], result["night"]
print(f"  Wives:          {0.30*d + 0.30*n:.1f} (need >= 150)")
print(f"  Husbands:       {0.10*d + 0.30*n:.1f} (need >= 120)")
print(f"  Single Males:   {0.10*d + 0.15*n:.1f} (need >= 100)")
print(f"  Single Females: {0.10*d + 0.20*n:.1f} (need >= 110)")
print(f"  Night ratio:    {n/(d+n)*100:.1f}% (limit 50%)")

HiGHS 1.11.0=== TELEPHONE SURVEY RESULTS ===
Daytime calls   -> 900.00
Nighttime calls -> 100.00
Total calls     -> 1000.00
Minimum cost    -> $2300.00
Time            -> 0.01081076 seconds

Expected contacts:
  Wives:          300.0 (need >= 150)
  Husbands:       120.0 (need >= 120)
  Single Males:   105.0 (need >= 100)
  Single Females: 110.0 (need >= 110)
  Night ratio:    10.0% (limit 50%)


## Sensitivity Analysis

In [6]:
ampl = create_ampl_instance()

ampl.eval(
    r'''
    var day >= 0;
    var night >= 0;

    minimize TotalCost:
        2 * day + 5 * night;

    subject to Wives:
        0.30 * day + 0.30 * night >= 150;

    subject to Husbands:
        0.10 * day + 0.30 * night >= 120;

    subject to SingleMales:
        0.10 * day + 0.15 * night >= 100;

    subject to SingleFemales:
        0.10 * day + 0.20 * night >= 110;

    subject to NightLimit:
        night <= day;
    '''
)
ampl.solve()

constraints = ["Wives", "Husbands", "SingleMales", "SingleFemales", "NightLimit"]

print("=== SHADOW PRICES AND SLACK ===")
print(f"{'Constraint':<18} {'Shadow Price':>14} {'Slack':>10}")
print("-" * 44)
for c in constraints:
    con = ampl.get_constraint(c)
    print(f"{c:<18} {con.dual():>14.4f} {con.slack():>10.4f}")

print()
print("=== REDUCED COSTS ===")
print(f"day   RC = {ampl.get_variable('day').rc():.4f}")
print(f"night RC = {ampl.get_variable('night').rc():.4f}")

print()
print("=== MARGINAL COST PER ADDITIONAL PERSON ===")
for c in ["Wives", "Husbands", "SingleMales", "SingleFemales"]:
    dual = ampl.get_constraint(c).dual()
    if abs(dual) > 1e-8:
        print(f"  One more {c:<15}: ${abs(dual):.4f} additional cost")
    else:
        print(f"  One more {c:<15}: $0 (constraint not binding)")

HiGHS 1.11.0=== SHADOW PRICES AND SLACK ===
Constraint           Shadow Price      Slack
--------------------------------------------
Wives                      0.0000   150.0000
Husbands                  10.0000     0.0000
SingleMales                0.0000     5.0000
SingleFemales             10.0000     0.0000
NightLimit                 0.0000   800.0000

=== REDUCED COSTS ===
day   RC = 0.0000
night RC = 0.0000

=== MARGINAL COST PER ADDITIONAL PERSON ===
  One more Wives          : $0 (constraint not binding)
  One more Husbands       : $10.0000 additional cost
  One more SingleMales    : $0 (constraint not binding)
  One more SingleFemales  : $10.0000 additional cost


## Interpretation

The sensitivity analysis shows:

- **Shadow prices** on demographic constraints reveal the marginal cost of increasing each group's requirement by one person. The group with the highest shadow price is the most expensive demographic to reach.
- **Night limit**: If binding, the shadow price shows how much cost could be saved if more night calls were allowed. Night calls are more expensive ($5 vs $2) but have higher contact rates for husbands and single people.
- **Non-binding constraints** have zero shadow price — those demographics are already over-served by the optimal call mix.